# Tracking the Exact Global Phase with `PhasedOutcomeCompleteSimulation`

This notebook introduces `PhasedOutcomeCompleteSimulation`, the global-phase-resolving
generalization of `OutcomeCompleteSimulation`. It implements Algorithm 4.2 of
[arXiv:2603.24717](https://arxiv.org/abs/2603.24717), which extends the outcome-complete
stabilizer simulation of [arXiv:2309.08676](https://arxiv.org/abs/2309.08676) to keep track of
the **exact global phase** of the simulated state.

## Why the global phase matters

Ordinary stabilizer simulation only represents a state *up to a global phase*. That is enough for
sampling measurement outcomes, but **not** enough to verify equivalence of non-stabilizer circuits.
The motivating example from the paper is checking, for two pairs of Clifford unitaries
$(C_1, C_2)$ and $(D_1, D_2)$, whether

$$ C_1\, e^{i\alpha Z_1}\, C_2\,|0\rangle \;=\; D_1\, e^{i\alpha Z_1}\, D_2\,|0\rangle \quad\text{for all } \alpha. $$

This holds **iff** the stabilizer states $C_1 Z_1^a C_2 |0\rangle$ and $D_1 Z_1^a D_2 |0\rangle$ are
equal for $a \in \{0, 1\}$ — and that equality must be **exact, including the global phase**.
`PhasedOutcomeCompleteSimulation` is the engine that makes this exact comparison possible.

Crucially, none of this ever forms an exponentially large state vector: phases are tracked
symbolically as integer powers of $\zeta_8 = e^{i\pi/4}$, and every operation stays polynomial in the
number of qubits.

## Setup

In [1]:
from paulimer import (
    PhasedOutcomeCompleteSimulation,
    OutcomeCompleteSimulation,
    SparsePauli,
    UnitaryOpcode,
)

## The phased stabilizer-state representation

Like `OutcomeCompleteSimulation`, the phased simulator tracks **all** $2^{n_r}$ branches of a
circuit with $n_r$ random measurement outcomes at once. For a random-bit assignment
$r \in \{0,1\}^{n_r}$ the encoded state is

$$ i^{\langle p,\, r\rangle}\, (-1)^{\langle B r + s,\, r\rangle}\; R\,|A r\rangle, $$

where

- `R` is the phased state encoder (a Clifford unitary *with* its exact global phase),
- `A` = `sign_matrix` maps the random bits to a computational-basis input of `R`,
- `B` = `quadratic_phase_matrix` carries the quadratic $(-1)$-phase,
- `p` = `linear_i_phase` carries the linear $i$-phase,
- `s` = `linear_sign_phase` carries the linear $(-1)$-phase.

The scalar prefactor $i^{\langle p,r\rangle}(-1)^{\langle Br+s,r\rangle}$ is returned, as a
$\zeta_8$ exponent in $\{0,\dots,7\}$, by `output_phase_exponent(r)`.

## A worked example: interfering measurement phases

We prepare $|{+}{+}\rangle$ and measure the Pauli $Y$ on each qubit. Each $Y$ measurement is a random
outcome, so there are two random bits $r = (r_0, r_1)$ and four branches.

In [2]:
sim = PhasedOutcomeCompleteSimulation(2)
sim.apply_unitary(UnitaryOpcode.Hadamard, [0])
sim.apply_unitary(UnitaryOpcode.Hadamard, [1])
sim.measure(SparsePauli("Y_0"))
sim.measure(SparsePauli("Y_1"))

print("qubits           :", sim.qubit_count)
print("random outcomes  :", sim.random_outcome_count)
print("A (sign_matrix)  :", repr(sim.sign_matrix))
print("B (quad. phase)  :", repr(sim.quadratic_phase_matrix))
print("p (linear i)     :", sim.linear_i_phase)
print("s (linear sign)  :", sim.linear_sign_phase)

qubits           : 2
random outcomes  : 2
A (sign_matrix)  : 10
01

B (quad. phase)  : 01
00

p (linear i)     : [11]
s (linear sign)  : [00]


The linear $i$-phase is $p = (1, 1)$: each random $Y$ outcome contributes a factor of $i$.
The quadratic matrix $B$ has a single off-diagonal entry. Let us read off the $\zeta_8$ exponent of
every branch and translate it into a familiar scalar.

In [3]:
ZETA8_LABEL = {0: "1", 1: "ζ₈", 2: "i", 3: "ζ₈³", 4: "-1", 5: "ζ₈⁵", 6: "-i", 7: "ζ₈⁷"}

def branches(n_random):
    for k in range(2 ** n_random):
        yield [bool((k >> i) & 1) for i in range(n_random)]

for r in branches(sim.random_outcome_count):
    e = sim.output_phase_exponent(r)
    print(f"r = {tuple(int(b) for b in r)}  ->  zeta8^{e}  =  {ZETA8_LABEL[e]}")

r = (0, 0)  ->  zeta8^0  =  1
r = (1, 0)  ->  zeta8^2  =  i
r = (0, 1)  ->  zeta8^2  =  i
r = (1, 1)  ->  zeta8^4  =  -1


## The punchline

Look at the four phases: $1,\ i,\ i,\ -1$.

- Branches $(1,0)$ and $(0,1)$ each pick up a single $i$ from one $Y$ outcome.
- Branch $(1,1)$ picks up **both**, and the two factors of $i$ interfere to give
  $i \cdot i = -1$ (that is $\zeta_8^4$), *not* $i + i$.

This is exactly the kind of exact, non-trivial global phase that ordinary stabilizer simulation
discards. The phased simulator stores the linear $i$-phase $p$ **modulo 2**, so the $i\cdot i = -1$
carry is absorbed into the quadratic matrix $B$ — which is why $B$ is needed at all.

The plain `OutcomeCompleteSimulation` simply does not expose this information:

In [4]:
plain = OutcomeCompleteSimulation(2)
print("OutcomeCompleteSimulation has output_phase_exponent? ",
      hasattr(plain, "output_phase_exponent"))
print("OutcomeCompleteSimulation has linear_i_phase?        ",
      hasattr(plain, "linear_i_phase"))

OutcomeCompleteSimulation has output_phase_exponent?  False
OutcomeCompleteSimulation has linear_i_phase?         False


## Recomputing the phase from the raw data

To make the formula concrete, we recompute $i^{\langle p,r\rangle}(-1)^{\langle Br+s,r\rangle}$
directly from the exposed matrices and vectors — using only integer (GF(2)) arithmetic — and check
it against `output_phase_exponent` for every branch.

In [5]:
def phase_exponent_from_data(sim, r):
    """Reconstruct the zeta8 exponent from A, B, p, s using exact integer arithmetic."""
    B = sim.quadratic_phase_matrix
    p = sim.linear_i_phase
    s = sim.linear_sign_phase
    n = sim.random_outcome_count

    linear_i = sum(1 for i in range(n) if p[i] and r[i]) % 2

    quadratic = 0
    for i in range(n):
        for j in range(n):
            if B[i, j] and r[i] and r[j]:
                quadratic ^= 1
    linear_sign = sum(1 for i in range(n) if s[i] and r[i]) % 2
    sign = quadratic ^ linear_sign

    return (2 * linear_i + 4 * sign) % 8

for r in branches(sim.random_outcome_count):
    assert phase_exponent_from_data(sim, r) == sim.output_phase_exponent(r)
print("✓ Reconstructed phases match output_phase_exponent for all branches")

✓ Reconstructed phases match output_phase_exponent for all branches


## Why this enables circuit verification

Recall the equivalence $C_1 e^{i\alpha Z_1} C_2 |0\rangle = D_1 e^{i\alpha Z_1} D_2|0\rangle$
reduces to the **exact** equality of the stabilizer states $C_1 Z_1^a C_2 |0\rangle$ and
$D_1 Z_1^a D_2 |0\rangle$ for $a \in \{0,1\}$. Because `PhasedOutcomeCompleteSimulation` retains the
exact global phase of each branch's encoded state $R|Ar\rangle$ — rather than only the state up to a
phase — it provides precisely the information needed to decide such equalities. The same machinery
extends to circuits with intermediate measurements and outcome-parity-conditional Pauli gates, which
are ubiquitous in fault-tolerant quantum computing.

Everything above runs in time polynomial in the number of qubits: the simulator never materializes a
$2^n$-dimensional state vector, and the global phase is carried exactly as an integer $\zeta_8$
exponent. The implementation is validated phase-exactly against brute-force dense statevector
references in the Rust test suite (`pauliverse/tests/phased_outcome_complete_dense.rs`).